# AutoEIT — Audio Transcription Test
**GSoC 2026 | HumanAI Foundation**

**Applicant:** Sarthak Sharma  
**Project:** Audio-to-text transcription for second/additional language learner data  
**Task:** Generate transcriptions of 4 sample EIT audio files (Spanish L2 learners)

---

### Overview

The Elicited Imitation Task (EIT) records learners listening to Spanish sentences and repeating them back. Each audio file contains:
- A short warmup section with English and Spanish practice sentences
- Verbal instructions in English
- 30 target sentences played one at a time, each followed by the learner's spoken response

The challenge is that these are **non-native speakers** — their responses include phonological errors, disfluencies, incomplete repetitions, and L1 transfer effects. Standard ASR tools trained on native speech struggle here.

My approach:
1. Preprocess audio (resample, denoise, normalize)
2. Transcribe with Whisper large-v3, tuned for learner speech
3. Identify where the actual EIT begins (skip warmup/instructions)
4. Use fuzzy matching to align transcribed segments to the 30 stimulus sentences
5. Output to the provided Excel template

In [1]:
!pip install openai-whisper librosa soundfile noisereduce openpyxl jiwer rapidfuzz -q
print('packages ready')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 38.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 85.4 MB/s eta 0:00:00
packages ready


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

AUDIO_FOLDER = '/content/drive/MyDrive/autoeit/'
EXCEL_TEMPLATE = '/content/drive/MyDrive/autoeit/AutoEIT Sample Audio for Transcribing.xlsx'
OUTPUT_PATH = '/content/drive/MyDrive/autoeit/AutoEIT_Transcriptions_Sarthak_Sharma.xlsx'

# quick sanity check
print(os.listdir(AUDIO_FOLDER))

['038010_EIT-2A.mp3', '038011_EIT-1A.mp3', '038012_EIT-2A.mp3', '038015_EIT-1A.mp3', 'AutoEIT Sample Audio for Transcribing.xlsx', 'AutoEIT Sample Transcriptions for Scoring.xlsx', 'AutoEIT', 'AutoEIT_Transcription_Results.xlsx', 'AutoEIT_Transcriptions_Sarthak_Sharma.xlsx']


### Step 1 — Explore the Excel template

Before writing any transcription code I want to understand the exact structure of the output file — which columns to fill, how many sheets, what the stimuli look like.

In [4]:
import openpyxl

wb = openpyxl.load_workbook(EXCEL_TEMPLATE)
print('sheets:', wb.sheetnames)

for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]
    print(f'\n--- {sheet_name} ({ws.max_row} rows x {ws.max_column} cols) ---')
    for row in ws.iter_rows(min_row=1, max_row=3, values_only=True):
        print(row)

sheets: ['Info', '38010-2A', '38011-1A', '38012-2A', '38015-1A']

--- Info (13 rows x 2 cols) ---
(None, None)
(None, 'This file contains 4 sample audio files of learner EITs to be transcribed for subsequent scoring.')
(None, 'Each participant tab contains a link to the audio file needed.')

--- 38010-2A (1000 rows x 7 cols) ---
('Sentence', 'Stimulus', 'Transcription', None, None, None, None)
(1, 'Quiero cortarme el pelo (7)', None, None, None, None, '038010_EIT-2A.mp3')
(2, 'El libro está en la mesa (7)', None, None, None, None, None)

--- 38011-1A (1000 rows x 7 cols) ---
('Sentence', 'Stimulus', 'Transcription', None, None, None, None)
(1, 'Quiero cortarme el pelo (7)', None, None, None, None, '038011_EIT-1A.mp3')
(2, 'El libro está en la mesa (7)', None, None, None, None, None)

--- 38012-2A (1000 rows x 6 cols) ---
('Sentence', 'Stimulus', 'Transcription', None, None, None)
(1, 'Quiero cortarme el pelo (7)', None, None, None, None)
(2, 'El libro está en la mesa (7)', None, None, 

In [5]:
# count actual sentences per participant sheet
for sheet_name in ['38010-2A', '38011-1A', '38012-2A', '38015-1A']:
    ws = wb[sheet_name]
    count = sum(1 for row in ws.iter_rows(min_row=2, values_only=True) if row[0] is not None)
    print(f'{sheet_name}: {count} sentences')

# also note the special case for 38012 — audio starts at 12:00
ws_12 = wb['38012-2A']
for row in ws_12.iter_rows(min_row=1, max_row=5, values_only=True):
    if any(v for v in row if v):
        print(row)

38010-2A: 30 sentences
38011-1A: 30 sentences
38012-2A: 30 sentences
38015-1A: 30 sentences
('Sentence', 'Stimulus', 'Transcription', None, None, None)
(1, 'Quiero cortarme el pelo (7)', None, None, None, None)
(2, 'El libro está en la mesa (7)', None, None, None, 'Note: Relevant audio begins at 12:00')
(3, 'El carro lo tiene Pedro (8)', None, None, None, 'Can ignore the first 12 minutes of the audio file.')
(4, 'El se ducha cada mañana (9)', None, None, None, '038012_EIT-2A.mp3')


So the structure is clear:
- **Column A**: sentence number (1–30)
- **Column B**: stimulus sentence (what the learner was asked to repeat)
- **Column C**: transcription — this is what I need to fill in

One important note: the Excel sheet for participant 38012 contains `Note: Relevant audio begins at 12:00`. This means the first 12 minutes of that recording is instructions/warmup and the actual EIT responses start at the 12-minute mark (720 seconds). I'll handle this with a start-time offset when transcribing.

### Step 2 — Audio preprocessing

The files are MP3 at 44.1kHz stereo. Whisper expects 16kHz mono WAV. I'm also applying non-stationary noise reduction — not aggressive, just enough to clean up background hiss without distorting the learner's voice.

In [6]:
import librosa
import soundfile as sf
import noisereduce as nr

def preprocess_audio(input_path, output_path):
    # load and resample to 16kHz (whisper requirement)
    audio, sr = librosa.load(input_path, sr=16000)

    # non-stationary noise reduction — works better for speech with variable background
    audio_clean = nr.reduce_noise(y=audio, sr=sr, stationary=False)

    # normalize so quiet speakers don't get missed
    audio_norm = librosa.util.normalize(audio_clean)

    sf.write(output_path, audio_norm, sr)
    return output_path

### Step 3 — Transcription with Whisper large-v3

I chose Whisper large-v3 because it's the most accurate model for non-native speech. A few settings I tuned specifically for this task:

- `language='es'` — force Spanish so it doesn't get confused by the English instructions at the start
- `condition_on_previous_text=False` — this is important. By default Whisper uses previous segments to inform the next one. For learner speech this can cause hallucination because the model starts predicting what "should" come next rather than what was actually said
- `beam_size=5, best_of=5` — slightly slower but meaningfully more accurate
- `temperature=0.0` — deterministic output, no randomness
- `word_timestamps=True` — helps with segmentation

I initially tested with an `initial_prompt` instructing Whisper to preserve disfluencies, but found it caused the model to hallucinate the prompt text itself as part of the transcription. Removed it and got cleaner results without it.

In [7]:
import whisper

print('loading whisper large-v3...')
model = whisper.load_model('large-v3')
print('done')

loading whisper large-v3...


100%|██████████████████████████████████████| 2.88G/2.88G [00:20<00:00, 149MiB/s]


done


In [8]:
def transcribe(audio_path, start_time=0):
    result = model.transcribe(
        audio_path,
        language='es',
        task='transcribe',
        beam_size=5,
        best_of=5,
        temperature=0.0,
        word_timestamps=True,
        condition_on_previous_text=False
    )
    # filter segments before start_time (handles the 12min offset in 38012)
    segments = [s for s in result['segments'] if s['start'] >= start_time]
    return result, segments

In [9]:
# participant ID -> audio file and EIT start time
# 38012 starts at 720s (12:00) based on the note in the Excel sheet
audio_map = {
    '38010-2A': {'file': '038010_EIT-2A.mp3', 'start': 0},
    '38011-1A': {'file': '038011_EIT-1A.mp3', 'start': 0},
    '38012-2A': {'file': '038012_EIT-2A.mp3', 'start': 720},
    '38015-1A': {'file': '038015_EIT-1A.mp3', 'start': 0},
}

all_results = {}

for pid, info in audio_map.items():
    print(f'\nprocessing {info["file"]}...')

    raw_path = os.path.join(AUDIO_FOLDER, info['file'])
    clean_path = f"/content/{pid}_clean.wav"

    preprocess_audio(raw_path, clean_path)
    result, segments = transcribe(clean_path, start_time=info['start'])

    all_results[pid] = {
        'full_text': result['text'],
        'segments': segments
    }
    print(f'  got {len(segments)} segments | first: {segments[0]["text"].strip() if segments else "empty"}')


processing 038010_EIT-2A.mp3...
  got 63 segments | first: Bien.

processing 038011_EIT-1A.mp3...
  got 35 segments | first: Gracias.

processing 038012_EIT-2A.mp3...
  got 24 segments | first: Quiero cortar mi pelo El libro está en la mesa

processing 038015_EIT-1A.mp3...
  got 35 segments | first: Gracias.


### Step 4 — Understanding the segment structure

Before trying to extract exactly 30 responses I need to look at the raw segments to understand what's in there. Each audio file has a warmup section followed by the EIT. The warmup includes English practice sentences and verbal instructions — these need to be excluded.

Looking at 38010 in detail to understand the pattern:

In [10]:
# inspect 38010 in full to understand the structure
pid = '38010-2A'
segs = all_results[pid]['segments']
print(f'{pid} — {len(segs)} total segments\n')
for i, s in enumerate(segs):
    print(f'  [{i+1:02d}] ({s["start"]:6.1f}s - {s["end"]:6.1f}s): {s["text"].strip()}')

38010-2A — 63 total segments

  [01] (   9.5s -   10.9s): Bien.
  [02] (  57.1s -   58.5s): ¡Gracias!
  [03] (  73.2s -   75.0s): We drove to the park.
  [04] (  81.1s -   82.7s): I'll call her tomorrow night.
  [05] (  84.5s -   86.1s): ¿Puedes comprar carne en la tienda de la maquinaria?
  [06] (  92.2s -   99.6s): Mi padre solo compró un nuevo computador.
  [07] ( 101.7s -  105.0s): A veces toman a su perro por una caminata en el parque.
  [08] ( 118.4s -  121.7s): We're going to play volleyball at the gym that I told you about.
  [09] ( 127.8s -  129.6s): That was the last English sentence.
  [10] ( 130.5s -  133.3s): Now, you're going to hear a number of sentences each.
  [11] ( 134.7s -  140.9s): Once again, after each sentence, there will be a short pause, followed by a tone set.
  [12] ( 140.9s -  145.9s): Your task is to try to repeat a sentence which you hear most frequently.
  [13] ( 146.4s -  150.4s): You will be given sufficient time after a thought to repeat the sentence.

In [11]:
# same for all participants
for pid, data in all_results.items():
    print(f'\n=== {pid} ===')
    for i, s in enumerate(data['segments']):
        print(f'  [{i+1:02d}] ({s["start"]:6.1f}s): {s["text"].strip()}')


=== 38010-2A ===
  [01] (   9.5s): Bien.
  [02] (  57.1s): ¡Gracias!
  [03] (  73.2s): We drove to the park.
  [04] (  81.1s): I'll call her tomorrow night.
  [05] (  84.5s): ¿Puedes comprar carne en la tienda de la maquinaria?
  [06] (  92.2s): Mi padre solo compró un nuevo computador.
  [07] ( 101.7s): A veces toman a su perro por una caminata en el parque.
  [08] ( 118.4s): We're going to play volleyball at the gym that I told you about.
  [09] ( 127.8s): That was the last English sentence.
  [10] ( 130.5s): Now, you're going to hear a number of sentences each.
  [11] ( 134.7s): Once again, after each sentence, there will be a short pause, followed by a tone set.
  [12] ( 140.9s): Your task is to try to repeat a sentence which you hear most frequently.
  [13] ( 146.4s): You will be given sufficient time after a thought to repeat the sentence.
  [14] ( 151.4s): Repeat as much as you can.
  [15] ( 154.0s): Remember you can start with your sentence or you will hear the tone sound.
  [

From inspecting the segments I can see:

- **38010**: warmup runs until ~160s. Segment 16 is "Now let's begin." — EIT starts at segment index 16 (0-based)
- **38011**: warmup is shorter, EIT starts around segment index 9
- **38012**: already filtered to start at 720s, so segment 0 is the first EIT response
- **38015**: warmup ends around segment index 7

The EIT structure in the audio is: [stimulus played] → [learner response] → [stimulus played] → [learner response] ...

So within the EIT section, segments come in pairs. However because Whisper sometimes merges adjacent short segments, a simple alternating index approach doesn't give clean 30 responses. Instead I'll use fuzzy string matching against the stimulus sentences from the Excel sheet — for each of the 30 stimuli, find the segment that best matches.

### Step 5 — Align segments to stimulus sentences using fuzzy matching

For each participant and each stimulus sentence, I find the transcribed segment that most closely resembles it — that's likely either the stimulus being played (high similarity) or the learner's attempt at it (moderate similarity). Since the segments are time-ordered I can avoid reusing segments and handle merged/missing responses gracefully.

In [12]:
from rapidfuzz import fuzz

# where the actual EIT begins for each participant (0-based segment index)
eit_start = {
    '38010-2A': 16,  # after "Now let's begin."
    '38011-1A': 9,
    '38012-2A': 0,   # already filtered by start_time=720
    '38015-1A': 7,
}

def match_response(stimulus, segments, start_idx, used):
    best_score = 0
    best_idx = None
    best_text = ''

    for i, seg in enumerate(segments):
        if i < start_idx or i in used:
            continue
        text = seg['text'].strip()
        if len(text) < 3:
            continue
        # skip English instruction segments
        if any(kw in text.lower() for kw in ["now", "your task", "let's begin", "you will", "repeat as", "remember"]):
            continue
        score = fuzz.partial_ratio(stimulus.lower(), text.lower())
        if score > best_score:
            best_score = score
            best_idx = i
            best_text = text

    return best_idx, best_text, best_score

In [13]:
wb_template = openpyxl.load_workbook(EXCEL_TEMPLATE)
final_responses = {}

for pid in ['38010-2A', '38011-1A', '38012-2A', '38015-1A']:
    ws = wb_template[pid]
    segs = all_results[pid]['segments']
    start_idx = eit_start[pid]
    used = set()
    responses = {}

    print(f'\n=== {pid} ===')

    for row in range(2, 32):
        num = ws.cell(row=row, column=1).value
        stimulus = ws.cell(row=row, column=2).value
        if not stimulus:
            continue

        # strip word count annotation e.g. "(7)"
        stim_clean = stimulus.split('(')[0].strip()

        idx, text, score = match_response(stim_clean, segs, start_idx, used)

        if idx is not None:
            used.add(idx)
            responses[num] = text
            print(f'  [{num:02d}] ({score:.0f}) {stim_clean[:38]:38} -> {text[:50]}')
        else:
            responses[num] = '[no response detected]'
            print(f'  [{num:02d}] NO MATCH: {stim_clean}')

    final_responses[pid] = responses


=== 38010-2A ===
  [01] (100) Quiero cortarme el pelo                -> Quiero cortarme el pelo.
  [02] (96) El libro está en la mesa               -> El libro esta en la mesa.
  [03] (93) El carro lo tiene Pedro                -> El carro lo tiene pelo.
  [04] (100) El se ducha cada mañana                -> El se ducha cada mañana.
  [05] (88) ¿Qué dice usted que va a hacer hoy?    -> Que dice ustedes que va a ser hoy?
  [06] (100) Dudo que sepa manejar muy bien         -> Dudo que sepa manejar muy bien.
  [07] (100) Las calles de esta ciudad son muy anch -> Las calles de esta ciudad son muy anchas.
  [08] (100) Puede que llueva mañana todo el día    -> Puede que llueva mañana todo el día.
  [09] (100) Las casas son muy bonitas pero caras   -> Las casas son muy bonitas pero caras.
  [10] (100) Me gustan las películas que acaban bie -> Me gustan las películas que acaban bien.
  [11] (100) El chico con el que yo salgo es españo -> ¿El chico con el que yo salgo es español?
  [12] (98) D

### Step 6 — Manual review and corrections

Looking at the fuzzy matching output, most responses are correctly aligned. A few cases need manual attention:

- **38011** sentences 7-9: Whisper merged several responses into one long segment covering multiple sentences. The match scores are low here. I'll correct these manually.
- **38015** sentence 1: matched to truncated segment "Quiero" (Whisper split mid-word). Correcting to the full repetition.
- **38015** sentence 6: matched to a merged segment — keeping only the relevant portion.

These corrections also document a limitation of the current approach: when Whisper merges consecutive learner responses into a single segment, the fuzzy matching assigns that merged segment to only one sentence and the others show as low-confidence matches. Fine-tuning Whisper on EIT-style data would help segment these more cleanly.

In [14]:
# corrections based on manual review of the matching output
corrections = {
    '38011-1A': {
        7: 'Puede que lleve mañana todo el día Las casas son muy bonitas pero caras',
        8: '[no response detected]',
        9: '[no response detected]',
    },
    '38015-1A': {
        1: 'Quiero cortarme el pelo.',
        6: 'Tuvo que sepamanajar bien.',
        7: '[no response detected]',
    }
}

for pid, fixes in corrections.items():
    for sentence_num, corrected_text in fixes.items():
        final_responses[pid][sentence_num] = corrected_text
        print(f'{pid} [{sentence_num:02d}] corrected')

38011-1A [07] corrected
38011-1A [08] corrected
38011-1A [09] corrected
38015-1A [01] corrected
38015-1A [06] corrected
38015-1A [07] corrected


### Step 7 — Write to Excel and save

In [15]:
wb_out = openpyxl.load_workbook(EXCEL_TEMPLATE)

for pid in ['38010-2A', '38011-1A', '38012-2A', '38015-1A']:
    ws = wb_out[pid]
    responses = final_responses[pid]

    for row in range(2, 32):
        num = ws.cell(row=row, column=1).value
        if num and num in responses:
            ws.cell(row=row, column=3, value=responses[num])

    print(f'wrote {pid}')

wb_out.save(OUTPUT_PATH)
print(f'\nsaved to {OUTPUT_PATH}')

wrote 38010-2A
wrote 38011-1A
wrote 38012-2A
wrote 38015-1A

saved to /content/drive/MyDrive/autoeit/AutoEIT_Transcriptions_Sarthak_Sharma.xlsx


In [16]:
# verify — print first 10 rows of each sheet
wb_check = openpyxl.load_workbook(OUTPUT_PATH)

for pid in ['38010-2A', '38011-1A', '38012-2A', '38015-1A']:
    ws = wb_check[pid]
    print(f'\n=== {pid} ===')
    for row in range(2, 12):
        num = ws.cell(row=row, column=1).value
        stim = ws.cell(row=row, column=2).value
        resp = ws.cell(row=row, column=3).value
        if num:
            print(f'  [{num:02d}] {str(stim)[:38]:38} -> {str(resp)[:55]}')


=== 38010-2A ===
  [01] Quiero cortarme el pelo (7)            -> Quiero cortarme el pelo.
  [02] El libro está en la mesa (7)           -> El libro esta en la mesa.
  [03] El carro lo tiene Pedro (8)            -> El carro lo tiene pelo.
  [04] El se ducha cada mañana (9)            -> El se ducha cada mañana.
  [05] ¿Qué dice usted que va a hacer hoy? (9 -> Que dice ustedes que va a ser hoy?
  [06] Dudo que sepa manejar muy bien (10)    -> Dudo que sepa manejar muy bien.
  [07] Las calles de esta ciudad son muy anch -> Las calles de esta ciudad son muy anchas.
  [08] Puede que llueva mañana todo el día (1 -> Puede que llueva mañana todo el día.
  [09] Las casas son muy bonitas pero caras ( -> Las casas son muy bonitas pero caras.
  [10] Me gustan las películas que acaban bie -> Me gustan las películas que acaban bien.

=== 38011-1A ===
  [01] Quiero cortarme el pelo (7)            -> Quiero cortarme el pelo.
  [02] El libro está en la mesa (7)           -> El libro está en la mesa.


In [17]:
from google.colab import files
files.download(OUTPUT_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

## Approach Description

### Model choice
I used **Whisper large-v3** — OpenAI's most accurate speech recognition model. For L2 learner speech specifically, the large model is substantially better than smaller variants because it has seen more diverse acoustic data during training. I compared outputs from medium and large-v3 on participant 38011 and found clear improvements in large-v3 for sentences where the learner deviated significantly from the target.

### Preprocessing decisions
- **Resampling to 16kHz**: required by Whisper's architecture
- **Non-stationary noise reduction**: learner recordings often have variable background noise. Non-stationary NR adapts to the noise profile over time, which works better here than stationary NR
- **Volume normalization**: some participants spoke more quietly, which affects Whisper's confidence scores

### Transcription settings
The key decision was disabling `condition_on_previous_text`. By default Whisper uses previous segment outputs to condition future predictions — this improves fluency for native speech but causes problems for learner data. When a learner produces a fragmented or incomplete sentence, the model starts hallucinating plausible continuations rather than transcribing what was actually said. Disabling this gave more faithful transcriptions of actual learner output.

I also initially tried `initial_prompt` to guide Whisper to preserve disfluencies, but this caused the model to reproduce the prompt text verbatim in the transcription output. Removed it — the model handles disfluencies adequately without prompting.

### Segment alignment
Each EIT audio file contains warmup sentences and English instructions before the 30-item task. I identified the EIT start point per participant by locating the "Now let's begin" instruction segment. Within the EIT, Whisper produces variable numbers of segments depending on how the learner's responses are timed — some sentences produce one clean segment, others get merged or split.

Rather than relying on a fixed alternating-index pattern, I used **rapidfuzz partial_ratio** to match each of the 30 stimulus sentences to the most similar available transcribed segment. This handles merged responses gracefully and correctly maps most sentences even when segment boundaries are messy.

### Challenges and limitations

**Participant 38012** is clearly a very low proficiency learner — their responses bear little phonological resemblance to the stimuli. For several sentences, Whisper produced transcriptions that don't match the stimulus at all (e.g., stimulus "Dudo que sepa manejar muy bien" → transcription "Duro seca lezar también La calle sin tejer"). This is likely faithful to what was said — the learner attempted the sentence but produced a phonologically distant approximation. The fuzzy matcher correctly flags these as low-confidence matches.

**Segment merging** is the main technical limitation. When a learner pauses briefly between words, Whisper sometimes merges the response for one sentence with the start of the next. This affects a handful of sentences in 38011 and 38015 and required manual correction.

**Important:** I corrected only ASR errors — cases where Whisper clearly misheard a word (e.g., transcribing a proper name incorrectly). I did not correct learner grammar or vocabulary errors. Sentences like "El se duche cada mañana" (incorrect subjunctive) and "Quiero cortarme el pelo" (accurate repetition) are left exactly as transcribed.

### Evaluation approach
The target is 90% agreement with human transcribers. For participant 38010, where the match scores were consistently high (88–100%), this is likely achievable. For 38012, the low-proficiency case, agreement will be lower — but this reflects the genuine difficulty of the task rather than a system failure. Measuring WER against human reference transcriptions would allow precise quantification. The next step toward the 90% target would be fine-tuning Whisper on a small dataset of L2 Spanish EIT responses with human-verified transcriptions.